<a href="https://colab.research.google.com/github/chrismji/Datasci-and-Ai-in-Bus/blob/main/T8_Regression_Regularization_Ames.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Regression and Regularization


We will go through 2 examples, one for linear regression and one for Logistic Regression (classification) - demonstrating Lasso and Ridge regularization.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso, Ridge, LinearRegression
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_percentage_error
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
%matplotlib inline



## Input and EDA: Ames Housing Data

We are going to use the Housing data colleced from Ames, Iowa.  This data has features on homes for sale and their selling price, we want to see if we can build a model to estimate sales price, which will be useful for looking for over or under-valued homes.  Data can be [downloaded from this link](https://drive.google.com/uc?export=download&id=1fq_hzpV0DbXAOqOZFndIKq-bsEO6JG9r).  More info about the data from the [Kaggle page](https://www.kaggle.com/datasets/shashanknecrothapa/ames-housing-dataset) or the [data dictionary](https://jse.amstat.org/v19n3/decock/DataDocumentation.txt).

The data set has a lot of features!!! For simplicity, I have selected a subset that are most interesting for this exercise, but feel free to explore others and see how good your model can get.

In [ ]:
# Input Ames data

df = pd.read_csv('AmesHousing.csv')

# take a look at the features and select the ones that we want to use
# df.columns

my_features = ['Lot Area','Year Built','Bldg Type', 'House Style','1st Flr SF','2nd Flr SF', 'Full Bath', 'TotRms AbvGrd', 'Bedroom AbvGr', 'Garage Area', 'Gr Liv Area',
       'Overall Qual','BsmtFin Type 1', 'BsmtFin SF 1','Heating', 'Heating QC',
       'Overall Cond','SalePrice']

Ames = df[my_features].copy()


# a little bit of feature engineering...creating a Total SqFt from 1st and 2nd
Ames['TotSqFt']=Ames['1st Flr SF']+Ames['2nd Flr SF']+Ames['BsmtFin SF 1']

# indicator for does it have a second floor?
Ames['Second']=np.where(Ames['2nd Flr SF']>0,1,0)
Ames.drop(['1st Flr SF','2nd Flr SF'],axis=1,inplace=True)

Ames.head()


In [ ]:
Ames.describe().round(2)

In [ ]:
Ames.hist(figsize=(15,15),edgecolor='black',bins=20)
plt.show()

In [ ]:
# replace Lot Area and TotSqFt with log versions

Ames['Lot Area']=np.log(Ames['Lot Area'])
Ames['TotSqFt']=np.log(Ames['TotSqFt'])


Check for and deal with NAs...

In [ ]:
# count missing values
Ames.isna().sum()

In [ ]:
# fill in NAs for BsmtMin Type1 with null, and remove others
Ames['BsmtFin Type 1'] = Ames['BsmtFin Type 1'].fillna('None')
Ames = Ames.dropna()

Because we are doing regression, we need to `get_dummies` for the categoricals (make sure to use `drop_first=True` and `dtype=int` because regression algorithms need numerics.

In [ ]:
Ames=pd.get_dummies(Ames,drop_first=True, dtype=int)
Ames.head()

In [ ]:
Ames.info()

In [ ]:
# define our X and y
X=Ames.drop(["SalePrice"],axis=1)
y=Ames["SalePrice"]

## Fit Regression Model

First fit a standard OLS model (OLS=_Ordinary Least Squares_ and is equivalent to Multiple Linear Regression)

I like the `OLS` module from `statsmodels.api` because it provides a nice regression output table, but `sklearn.linear_model import LinearRegression` would work fine here also.



In [ ]:
from statsmodels.api import OLS


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=99)


# unfortunately, it requires you to add a constant feature to run a regression
X_train_ols = sm.add_constant(X_train)
mlr_model=OLS(y_train,X_train_ols).fit()

# Print summary table
print(mlr_model.summary())

What can we learn from the regression table:
- What is the effect on price of having a Second Floor?
- What are the significant features? (recall that those with p-value (or P>|t|) less than 0.05 are typically labelled as significant.  Which features are significant??)


The $R^2$ looks good, but remember that the
 table above is all based on the training set, lets calculate an RMSE on the test set.

In [ ]:
# unfortunately, it requires you to add a constant feature to run a regression
X_test_ols = sm.add_constant(X_test)
y_pred=mlr_model.predict(X_test_ols)

mlr_rmse=root_mean_squared_error(y_pred,y_test)

print(round(mlr_rmse,1))


Can you interpret this RMSE value in the context of the problem? Can you also calculate the MAPE (mean absolute percentage error) and interpret that?

## Ridge Regression

Now lets see if ridge regression can do any better.  Remember that Ridge with $\alpha=0$ is the same as regular multiple regression (OLS) and should match the value above.


In [ ]:
alpha_best = 0
rmse_best = 100000

for a in np.arange(0,1000,50):
  model = Ridge(alpha=a, max_iter=10000) # what we called lambda in class is alpha in the Ridge function

  model.fit(X_train, y_train)
  y_pred=model.predict(X_test)
  rmse=root_mean_squared_error(y_pred,y_test)
  print("alpha=",a,": rmse = ",round(rmse,3))
  if rmse < rmse_best:
      rmse_best = rmse
      alpha_best = a
print("\nBest alpha = ",alpha_best,"with Best RMSE=",round(rmse_best,2))

What is the best value of $\alpha$?  What does this tell you?

## Lasso Regression

Lasso regularized regression is the same as Ridge, but with a different penalty (L1) based on absolute values.  Can we do better?

You do it this time - fill in the blanks to do the same thing we just did but for Lasso, and print out the best value of alpha and the RMSE:

In [ ]:
alpha_best = 0
rmse_best = 100000

for a in np.arange(###):
  model = Lasso(###) # what we called lambda in class is alpha in the Ridge function
  model.fit(###,###)
  y_pred=model.predict(###)
  rmse=root_mean_squared_error(###,###)
  print(###)
  if rmse < rmse_best:
      rmse_best = ###
      alpha_best = ###
print(###)

Looks like we might be able to do better with a little shrinkage!  What is the best across all methods?

Lets take this best method and look at the shrunken coefficients compared to the full (OLS) model.

To see the effect of the Lasso, print out the coefficients of the full regression model (no shrinkage) with the optimal Lasso model (with alpha_best).  What do you notice?

In [ ]:
model_noshrink = Lasso(alpha=0)
model_noshrink.fit(X_train,y_train)
model_noshrink.coef_

model_best = Lasso(alpha=alpha_best)
model_best.fit(X_train,y_train)
model_best.coef_

coef_table = zip(X.columns,model_best.coef_.round(2),model_noshrink.coef_.round(2))


coef_df = pd.DataFrame(coef_table, columns=['colname', 'coef_best','coef_noshrink'])

print(coef_df)

Note the coefficients that were shrunk ALL the way to zero.  This is one of the great features of the Lasso.  These are noise (useless) features and we basically are removing them by setting their values to zero.  And the model gets much better!  

# Logistic Regularization: Tayko Catalog Data

Lets take a look at the Tayko data from the Shmueli book: [download link here](https://drive.google.com/uc?download&id=1wo7x7PmnCJ5-79RZXJSIAa7eS8DdrX-y).

This is an online retail company that is trying to predict whether or not a customer will make a Purchase from catalog mailings and other customer attributes.  More information about the data [at the Kaggle page here](https://www.kaggle.com/datasets/roaaalhaddad/tayko-software-cataloger).

The features consist of a bunch of indicator features for catalogs they have sent out to customers (`source_*`).  The target feature is "Purchase".  



In [ ]:
tayko = pd.read_csv('Tayko.csv')
tayko.info()
tayko.describe()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X=tayko.drop(["sequence_number","Spending","Purchase"],axis=1)
# add constant term to X
X = sm.add_constant(X)
y=tayko["Purchase"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=99)



Lets fit Logistic Regression using the `Logit()` function from `statsmodels`.  This allows us to get a full coefficient table.

In [ ]:
## fit FULL model - Logistic Regression.

## Use statmodels so we can get the full logistic regression table.
logit_model = sm.Logit(y_train, X_train).fit()

# Print the full coefficients table
print(logit_model.summary())


y_pred_prob = logit_model.predict(X_test)

# Calculate AUC
auc = roc_auc_score(y_test, y_pred_prob)

print("\nAUC:", round(auc,4))



### Finding Best Models

Now lets look at Lasso and Ridge.  Because this is a classification task we are going to use AUC as our metric instead of RMSE.  



Here we will use the `LogisticRegression()` function because `Logit()` does not support shrinkage 😠.   The arguments to `LogisticRegression()` are a shrinkage parameter C (which is = 1/alpha) and a penalty (`l1` for Lasso and `l2` for Ridge).  We can use `GridSearchCV()` to search across shrinkage values for both Lasso and Ridge:

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression

param_grid = {'C': [.01,.1,1,5,10,20,50],
              'penalty':['l1','l2']}

model = LogisticRegression(solver='liblinear')

# next line makes sure the K-Folds are random and stratified (same % of + in each fold)

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=13)
best_model = GridSearchCV(model, param_grid, cv=cv5, scoring='roc_auc')

# Fit the GridSearchCV to the training data
best_model.fit(X_train, y_train)

print(best_model.best_params_)


# Calculate AUC on test data from bestmodel
y_pred_prob = best_model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_prob)
print("AUC=",auc)

### Stepwise Regression

In the lecture, we learned about stepwise regression, where you remove one feature at a time, until the model degrades.  Below we wrote a function to implement it for this case:

In [ ]:

# Start with all features
features = list(X_train.columns)
best_auc = 0
best_features = features.copy()

while len(features) > 0:

        # Fit model with current features
        model = sm.Logit(y_train, X_train[best_features]).fit(disp=0) # Use statsmodels for p-values
        y_pred = model.predict(X_test[best_features])
        auc = roc_auc_score(y_test, y_pred)

        # Find feature with highest p-value
        p_values = model.pvalues # Exclude intercept
        worst_feature_index = np.argmax(p_values)
        worst_feature = features[worst_feature_index]
        print(f"Worst feature: {worst_feature}")

        # fit model with feature removed, and check AUC
        features.remove(worst_feature)
        new_features = features.copy()
        model_new = sm.Logit(y_train, X_train[new_features]).fit(disp=0)
        y_pred_new = model_new.predict(X_test[new_features])
        new_auc = roc_auc_score(y_test, y_pred_new)
        print("New AUC" ,round(new_auc,3), round(best_auc,3))

        # print out results of this iteration
        if round(new_auc,3) >= round(best_auc,3):
            print ("REMOVE",worst_feature,round(new_auc,3),"\n")
            best_features = new_features.copy()
            best_auc = new_auc
        else:
            print ("Stop, do not remove",worst_feature)
            break  # Stop if no more features to remove


print("\nSelected Features:", best_features)
print("Best AUC:", round(best_auc,4))


In [ ]:
# Fit final model with selected features
final_model = sm.Logit(y_train, X_train[best_features]).fit(disp=0)
print(final_model.summary())

# Evaluate on test set
y_pred_final = final_model.predict(X_test[best_features])
auc_final = roc_auc_score(y_test, y_pred_final)

print("Final AUC:", round(auc_final,4))
# print("Final RMSE on Test Set:", rmse_final)